# NLU HOSTAGE — ai_1_nlu_v3_600

Notebook ini hanya memakai modul training bersama. Semua cell legacy telah dihapus.


In [1]:
from pathlib import Path
DATASET_FILENAME = "chat_dataset2 copy.csv"
NOTEBOOK_FOLDER = "ai_1_nlu_v3_600"
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != NOTEBOOK_FOLDER:
    candidate = NOTEBOOK_DIR / NOTEBOOK_FOLDER
    if candidate.is_dir():
        NOTEBOOK_DIR = candidate
DATASET_PATH = NOTEBOOK_DIR / "data" / DATASET_FILENAME
print(f"Dataset aktif: {DATASET_PATH}")


Dataset aktif: C:\Users\andyc\Documents\a_skripsi\training\prethesis\ai_1_nlu_v3_600\data\chat_dataset2 copy.csv


In [2]:
# MODUL BERSAMA: EDA + SVM + SVM TUNING + NB + NB TUNING + TRANSFORMER
from pathlib import Path
import sys

PROJECT_ROOT = Path(NOTEBOOK_DIR).parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from modules.nlu_eda import run_nlu_eda
from modules.nlu_training import (
    predict_intent as _predict_intent_shared,
    predict_transformer_intent as _predict_transformer_intent_shared,
    run_all_nlu_models,
    train_naive_bayes,
    train_naive_bayes_tuned,
    train_svm,
    train_svm_tuned,
    train_transformer,
)

MODEL_DIR = Path(NOTEBOOK_DIR) / "models"
# Kebijakan proyek: SVM/NB pada CPU, Transformer pada GPU CUDA.
CLASSICAL_DEVICE = "cpu"
TRANSFORMER_DEVICE = "cuda"


def run_eda(plot=True):
    return run_nlu_eda(DATASET_PATH, plot=plot)

def train_svm_model():
    return train_svm(DATASET_PATH, MODEL_DIR)

def train_svm_tuned_model():
    return train_svm_tuned(DATASET_PATH, MODEL_DIR)

def train_naive_bayes_model():
    return train_naive_bayes(DATASET_PATH, MODEL_DIR)

def train_naive_bayes_tuned_model():
    return train_naive_bayes_tuned(DATASET_PATH, MODEL_DIR)

def train_transformer_model(epochs=4):
    return train_transformer(DATASET_PATH, MODEL_DIR, epochs=epochs, device=TRANSFORMER_DEVICE)

def predict_intent(text, model_filename=None):
    return _predict_intent_shared(text, MODEL_DIR, model_filename)

def predict_transformer_intent(text):
    return _predict_transformer_intent_shared(text, MODEL_DIR, device=TRANSFORMER_DEVICE)

HOSTAGE_TEST_CASES = [
    ("accusing", "B kena Gag Order saat menjelaskan alibi, menurut gw itu pola Hitman."),
    ("defending", "Gw bukan Hitman, tuduhan itu gak punya bukti publik."),
    ("bluffing", "Gw Spy, semalam gw Guard Raka dan dia pasti aman."),
    ("probing", "Stalker, semalam lu Peek siapa dan hasilnya apa?"),
    ("deflecting", "Jangan fokus ke gw, cek D yang terus mengubah cerita tiap ditanya."),
    ("persuading", "Vote C aja, dia paling diuntungkan dari korban Hostage semalam."),
    ("claiming", "Klaim gw Civilian, gw gak punya skill malam."),
    ("neutral", "Fase malam bikin chat terkunci, kita tunggu pagi dulu."),
]

def run_hostage_test_suite(model_filename=None):
    correct = 0
    for expected, chat in HOSTAGE_TEST_CASES:
        predicted, confidence = predict_intent(chat, model_filename)
        correct += predicted == expected
        print(f"{expected:12} | prediksi={predicted:12} | confidence={confidence:6.2f}% | {chat}")
    print(f"\nCocok: {correct}/{len(HOSTAGE_TEST_CASES)}")

print("Modul NLU siap. Jalankan: run_eda(), train_svm_model(), train_svm_tuned_model(),")
print("train_naive_bayes_model(), train_naive_bayes_tuned_model(), atau train_transformer_model().")
print("Mode training aktif: CPU untuk SVM/Naive Bayes, GPU CUDA untuk Transformer.")
print("Model tersimpan terpisah; prediksi default memprioritaskan SVM tuned.")


Modul NLU siap. Jalankan: run_eda(), train_svm_model(), train_svm_tuned_model(),
train_naive_bayes_model(), train_naive_bayes_tuned_model(), atau train_transformer_model().
Mode training aktif: CPU untuk SVM/Naive Bayes, GPU CUDA untuk Transformer.
Model tersimpan terpisah; prediksi default memprioritaskan SVM tuned.


In [3]:
# JALANKAN SEMUA MODEL: empat model CPU, lalu Transformer CUDA dan 10 chat uji.
RUN_TRANSFORMER = True
RUN_TUNING = False
TRANSFORMER_EPOCHS = 4
artifacts, hasil_training, hasil_manual_test = run_all_nlu_models(
    DATASET_PATH, MODEL_DIR,
    run_transformer=RUN_TRANSFORMER,
    run_tuning=RUN_TUNING,
    transformer_epochs=TRANSFORMER_EPOCHS,
    transformer_device=TRANSFORMER_DEVICE,
)
print('RINGKASAN EVALUASI HOLDOUT:')
display(hasil_training)
print('RINGKASAN 10 CHAT UJI:')
display(hasil_manual_test)



MENJALANKAN: SVM baseline
SVM | train=4476 | test=1120 | kelas=8



--- Evaluasi SVM baseline (holdout test set) ---
Accuracy    : 0.7446
Macro F1    : 0.7462
Weighted F1 : 0.7438
              precision    recall  f1-score   support

    accusing       0.71      0.70      0.71       144
    bluffing       0.60      0.59      0.60       144
    claiming       0.63      0.66      0.64       145
   defending       0.69      0.66      0.67       138
  deflecting       0.73      0.76      0.74       132
     neutral       0.96      0.94      0.95       140
  persuading       0.71      0.69      0.70       144
     probing       0.93      0.98      0.96       133

    accuracy                           0.74      1120
   macro avg       0.75      0.75      0.75      1120
weighted avg       0.74      0.74      0.74      1120

Model tersimpan: C:\Users\andyc\Documents\a_skripsi\training\prethesis\ai_1_nlu_v3_600\models\intent_classifier_svm.pkl

MENJALANKAN: Naive Bayes baseline
Naive Bayes | train=4476 | test=1120 | kelas=8



--- Evaluasi Naive Bayes baseline (holdout test set) ---
Accuracy    : 0.7107
Macro F1    : 0.7139
Weighted F1 : 0.7114
              precision    recall  f1-score   support

    accusing       0.65      0.72      0.68       144
    bluffing       0.55      0.65      0.60       144
    claiming       0.60      0.61      0.60       145
   defending       0.66      0.54      0.60       138
  deflecting       0.73      0.77      0.75       132
     neutral       0.95      0.89      0.92       140
  persuading       0.69      0.62      0.65       144
     probing       0.91      0.91      0.91       133

    accuracy                           0.71      1120
   macro avg       0.72      0.71      0.71      1120
weighted avg       0.72      0.71      0.71      1120

Model tersimpan: C:\Users\andyc\Documents\a_skripsi\training\prethesis\ai_1_nlu_v3_600\models\intent_classifier_nb.pkl

MENJALANKAN: IndoBERT Transformer


C:\Users\andyc\Documents\a_skripsi\training\prethesis\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Map:   0%|          | 0/4476 [00:00<?, ? examples/s]

...

expected=persuading   | predicted=accusing     | confidence= 53.11% | MISS
expected=claiming     | predicted=claiming     | confidence= 46.89% | OK
expected=neutral      | predicted=neutral      | confidence= 50.27% | OK
expected=accusing     | predicted=deflecting   | confidence= 49.90% | MISS
expected=defending    | predicted=neutral      | confidence= 36.54% | MISS

--- 10 chat uji: IndoBERT Transformer ---


expected=accusing     | predicted=accusing     | confidence= 93.85% | OK
expected=defending    | predicted=defending    | confidence= 95.77% | OK
expected=bluffing     | predicted=bluffing     | confidence= 61.62% | OK
expected=probing      | predicted=probing      | confidence= 99.08% | OK
expected=deflecting   | predicted=deflecting   | confidence= 96.43% | OK
expected=persuading   | predicted=persuading   | confidence= 93.36% | OK
expected=claiming     | predicted=defending    | confidence= 52.32% | MISS
expected=neutral      | predicted=neutral      | confidence= 98.38% | OK
expected=accusing     | predicted=accusing     | confidence= 90.29% | OK
expected=defending    | predicted=defending    | confidence= 97.63% | OK
RINGKASAN EVALUASI HOLDOUT:


,model,accuracy_holdout,macro_f1_holdout,weighted_f1_holdout,waktu_detik,status
0,IndoBERT Transformer,0.8205,0.8223,0.8201,139.0,berhasil
1,SVM baseline,0.7446,0.7462,0.7438,0.6,berhasil
2,Naive Bayes baseline,0.7107,0.7139,0.7114,0.3,berhasil


RINGKASAN 10 CHAT UJI:


,model,benar_dari_10,akurasi_10_chat
0,IndoBERT Transformer,9,0.9
1,SVM baseline,8,0.8
2,Naive Bayes baseline,7,0.7


## Laporan eksekusi notebook

Tuning SVM dan Naive Bayes dilewati untuk mempercepat run ini. Output training lengkap tersimpan pada cell tepat di atas.

| Model | Macro-F1 holdout | Uji 10 chat |
|---|---:|---:|
| SVM baseline | 0.7462 | 8/10 |
| Naive Bayes baseline | 0.7139 | 7/10 |
| IndoBERT Transformer (GPU) | 0.8223 | 9/10 |

v3_600 adalah hasil paling kuat saat ini; Transformer direkomendasikan sebagai kandidat utama.